In [ ]:
!pip install gymnasium

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import gymnasium as gym

np.random.seed(17)
tf.random.set_seed(17)

In [ ]:
env = gym.make("CartPole-v1")

state, _ = env.reset(seed=17)

In [ ]:
inputs = layers.Input(shape=(4,))
common = layers.Dense(64, activation="relu")(inputs)
common2 = layers.Dense(64, activation="relu")(common)

action = layers.Dense(2, activation="softmax")(common2)
critic = layers.Dense(1)(common2)

model = keras.Model(inputs=inputs, outputs=[action, critic])

In [ ]:
gamma = 0.995
optimizer = keras.optimizers.Adam(learning_rate=0.001)

eps = np.finfo(np.float32).eps.item()

running_reward = 0
episode_count = 0

In [ ]:
def compute_loss(action_prob, value, ret):
    advantage = ret - value

    actor_loss = -tf.math.log(action_prob + 1e-8) * advantage
    critic_loss = advantage**2

    return actor_loss + critic_loss

In [ ]:
while True:
    state, _ = env.reset()
    episode_reward = 0

    states = []
    actions = []
    rewards = []
    values = []

    with tf.GradientTape() as tape:
        for t in range(1, 1000):

            state_tensor = tf.expand_dims(tf.convert_to_tensor(state), 0)

            action_probs, critic_value = model(state_tensor)

            action = np.random.choice(2, p=np.squeeze(action_probs))

            states.append(state_tensor)
            actions.append(action)
            values.append(critic_value[0, 0])

            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            rewards.append(reward)
            episode_reward += reward

            if done:
                break

        running_reward = 0.05 * episode_reward + (1 - 0.05) * running_reward

        returns = []
        discounted_sum = 0
        for r in rewards[::-1]:
            discounted_sum = r + gamma * discounted_sum
            returns.insert(0, discounted_sum)

        returns = np.array(returns)
        returns = (returns - np.mean(returns)) / (np.std(returns) + eps)

        loss = 0
        for i in range(len(rewards)):
            action_probs, _ = model(states[i])
            prob = action_probs[0, actions[i]]

            loss += compute_loss(prob, values[i], returns[i])

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    episode_count += 1

    print(f"Эпизод {episode_count}: reward={episode_reward}, running_reward={running_reward:.2f}")

    if running_reward >= 195:
        print("\nСреда решена!")
        print("running_reward=195 достигнут на эпизоде:", episode_count)
        break

Эпизод 1: reward=42.0, running_reward=2.10
Эпизод 2: reward=10.0, running_reward=2.50
Эпизод 3: reward=16.0, running_reward=3.17
Эпизод 4: reward=20.0, running_reward=4.01
Эпизод 5: reward=10.0, running_reward=4.31
Эпизод 6: reward=11.0, running_reward=4.65
Эпизод 7: reward=21.0, running_reward=5.46
Эпизод 8: reward=31.0, running_reward=6.74
Эпизод 9: reward=49.0, running_reward=8.85
Эпизод 10: reward=54.0, running_reward=11.11
Эпизод 11: reward=12.0, running_reward=11.15
Эпизод 12: reward=29.0, running_reward=12.05
Эпизод 13: reward=54.0, running_reward=14.14
Эпизод 14: reward=11.0, running_reward=13.99
Эпизод 15: reward=19.0, running_reward=14.24
Эпизод 16: reward=19.0, running_reward=14.48
Эпизод 17: reward=14.0, running_reward=14.45
Эпизод 18: reward=13.0, running_reward=14.38
Эпизод 19: reward=18.0, running_reward=14.56
Эпизод 20: reward=15.0, running_reward=14.58
Эпизод 21: reward=14.0, running_reward=14.55
Эпизод 22: reward=21.0, running_reward=14.88
Эпизод 23: reward=17.0, runn

In [ ]:
state, _ = env.reset(seed=17)

state = tf.convert_to_tensor(state)
state = tf.expand_dims(state, 0)

action_probs, _ = model(state)

print("Вероятности действий:", action_probs.numpy())

Вероятности действий: [[0.4884701 0.5115299]]
